# Exploratory Analysis of Structural and Functional Features in SARS-CoV-2 RBD Variant FAIR² Datasets

This notebook performs an exploratory data analysis (EDA) of structural and functional metrics associated with SARS-CoV-2 receptor-binding domain (RBD) mutations, using datasets formatted with the FAIR² Croissant schema.

We analyze multiple RecordSets capturing AlphaFold and ESMFold predictions across variant sets, including 1-step and Omicron mutations. Features include RMSD, TM-score, SASA, binding affinity, expression levels, and Bio2Byte-derived descriptors, enabling visual inspection of patterns relevant to protein stability and ACE-2 interaction.


In [ ]:
# Install mlcroissant from the source
!sudo apt-get install python3-dev graphviz libgraphviz-dev pkg-config
!pip install mlcroissant==1.0.22
!pip install seaborn==0.13.2 tabulate==0.9.0 plotly==6.1.2

In [ ]:
import mlcroissant as mlc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

from IPython.display import Markdown, display


In [ ]:
# Load the dataset from a local path
url = "https://sen.science/doi/10.71728/hw56-vj34/fair2.json"

# Load the dataset from a local path
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")


## 2. Data Overview

In the **Croissant** format, a RecordSet represents a structured collection of records, where each record is a granular dataset unit (e.g., an image, text file, or table row). It defines the structure of these records using a set of fields, such as the columns in a table or sheet, as seen in this example.

### 2.1 Review available RecordSets

In [ ]:
# Format the list column as a Markdown-compatible string
def format_list_column(row):
    if isinstance(row, list):
        return "\n".join(f"- {item}" for item in row)  # Bullet point list
    return str(row)

In [ ]:
# List all the record sets available in the dataset
df = pd.DataFrame(metadata["recordSet"])
columns_to_keep = {
    "@id": "Record Set ID",
    "description": "Description"
}
df = df[list(columns_to_keep.keys())]
df = df.rename(columns=columns_to_keep)

# Convert DataFrame to Markdown table
markdown_table = tabulate(df, headers="keys", tablefmt="pipe", showindex=False)

# Render the table as Markdown in Jupyter
display(Markdown(markdown_table))

## 3. Data Extraction

#### 3.1 Load data from a specific record set into a DataFrame for analysis. 

In [ ]:
record_set_ids = [record["@id"] for record in metadata["recordSet"]]

dataframes = {
    record_set_id: pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    for record_set_id in record_set_ids
}

In [ ]:
prefix = "https://sen.science/doi/10.71728/hw56-vj34"

for name, df in dataframes.items():
    df.rename(columns=lambda x: x.replace(prefix, "").split("/")[-1], inplace=True)

In [ ]:
# Display the first rows of each dataframe
for name, df in dataframes.items():
    display(Markdown(f"#### {name}"))
    display(df.head())
    display(Markdown("---"))

## 4. Exploratory Data Analysis (EDA)

To grasp the dataset’s key characteristics, identify patterns, and detect anomalies, we begin with Exploratory Data Analysis (EDA).

### 4.2 Summary statistics



### 4.2.1 Observations from Summary Statistics

The following section provides a detailed analysis of the summary statistics derived from the dataset's record sets. These observations highlight key numerical features, central tendencies, variability, and potential outliers across various metrics. By examining these statistics, we can gain insights into the structural, functional, and energetic properties of the protein mutations, as well as their implications for binding affinity and expression levels. This foundational analysis sets the stage for deeper exploration and hypothesis generation.

In [ ]:
# Compute and display summary statistics for each recordset
for name, df in dataframes.items():
    display(Markdown(f"### Summary Statistics for RecordSet: {name}"))
    display(df.describe())
    display(Markdown("---"))

### 4.2.1 Observations from Summary Statistics

The summary statistics provide an overview of the key numerical features across the record sets. Below are the key observations:

1. **Central Tendency and Spread**:
    - Metrics such as `rmsd`, `tm_score`, and `sasa` exhibit varying ranges and means, reflecting differences in structural stability and solvent accessibility across mutations.
    - Binding-related metrics like `binding_energy` and `delta_bind` show variability, indicating the diverse impact of mutations on binding affinity.

2. **Hydrophobicity and Energy Metrics**:
    - `avg_hydro` and `total_hydro` provide insights into the hydrophobicity of the proteins, which is crucial for understanding folding and stability.
    - Energy metrics such as `local_net_energy` and `global_net_energy` highlight the energetic states of the proteins, with some mutations leading to significant changes.

3. **Structural and Functional Insights**:
    - Structural metrics like `plddt`, `average_interface_pae`, and `average_interface_plddt` indicate the confidence in structural predictions and interface stability.
    - Functional metrics such as `expr` and `delta_expr` reveal the expression levels and their changes, which are critical for understanding the functional impact of mutations.

4. **Outliers and Missing Data**:
    - Some fields, such as `bind`, `delta_bind`, and `expr`, have missing values, which may require further investigation or imputation.
    - Outliers in metrics like `binding_energy` and `confidence_bind` could indicate mutations with extreme effects, warranting closer examination.

These observations provide a foundation for deeper analysis, enabling the identification of patterns, anomalies, and relationships between mutations and their structural or functional consequences.

### 4.2.2 Distribution Analysis of Feature Values

This subsection focuses on analyzing the distribution of feature values across the dataset. By visualizing the distributions, we aim to identify patterns, variability, and potential outliers in the data. This analysis provides insights into the range and frequency of feature values, helping to understand the underlying characteristics of the dataset and guiding further exploratory and modeling efforts.

In [ ]:
# Select features to visualize
features_to_plot = ['rmsd', 'tm_score', 'sasa', 'plddt', 'expr']

# Plot histograms for each feature across all record sets
for feature in features_to_plot:
    plt.figure(figsize=(10, 6))
    for name, df in dataframes.items():
        label = name.split("/")[-1]  # Use only the last part after the "/"
        sns.histplot(df[feature].dropna(), kde=True, label=label, bins=30, alpha=0.5)
    plt.title(f"Distribution of {feature}")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.legend(title="Record Set")
    plt.show()

### 4.2.2 Observations from Distribution Analysis

The distribution analysis reveals key patterns across the selected features. Metrics such as `rmsd` and `tm_score` exhibit relatively narrow distributions, indicating consistent structural stability across mutations. In contrast, `sasa` shows a broader range, reflecting variability in solvent accessibility. The `plddt` feature demonstrates high confidence in structural predictions, with most values concentrated near the upper range. The `expr` feature, representing expression levels, displays a bimodal distribution, suggesting distinct groups of mutations with varying impacts on protein expression. These insights highlight the diversity in structural and functional effects of the analyzed mutations.

## 4.3 Correlation Analysis

This subsection focuses on analyzing the relationships between various features within the dataset using correlation matrices. By visualizing the correlations, we aim to identify patterns, dependencies, and potential redundancies among the features. This analysis provides insights into how different metrics interact, which can guide feature selection and model development for downstream tasks.

In [ ]:
# Compute and display correlations for each dataset using columns after 'rmsd'
for name, df in dataframes.items():
    display(Markdown(f"### Correlation Heatmap for RecordSet: {name}"))
    correlation_matrix = df.loc[:, 'rmsd':].corr()  # Select columns starting from 'rmsd'
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=False, cmap="coolwarm", cbar=True)
    plt.title(f"Correlation Heatmap for {name.split('/')[-1]}")
    plt.show()

### Observations from Correlation Analysis

The correlation heatmaps reveal several notable patterns across the dataset:

1. **Strong Positive Correlations**:
    - Features such as `iptm` and `iptm_ptm` exhibit near-perfect positive correlations, indicating redundancy or shared underlying factors.
    - Structural metrics like `backbone` and `helix` show strong positive correlations, reflecting their interdependence in protein stability.

2. **Negative Correlations**:
    - Features like `coil` and `backbone` display strong negative correlations, consistent with their opposing roles in protein structure.
    - `rmsd` shows weak negative correlations with metrics like `tm_score` and `plddt`, suggesting that higher structural stability corresponds to lower deviations.

3. **Weak or No Correlations**:
    - Metrics such as `binding_energy` and `confidence_bind` exhibit weak correlations with most other features, indicating their independence in capturing specific aspects of protein interactions.

4. **Clusters of Related Features**:
    - Hydrophobicity-related metrics (`avg_hydro`, `total_hydro`) form a distinct cluster, highlighting their shared influence on protein folding and stability.
    - Interaction metrics like `contact_pairs`, `hb`, and `sb` are moderately correlated, reflecting their collective role in protein-protein interactions.

These insights provide a foundation for feature selection and dimensionality reduction, aiding in downstream modeling and analysis.

### 4.4 Visualization of PDB Files

This section focuses on visualizing the structural data in PDB (Protein Data Bank) format. By rendering the 3D structures of proteins, we can gain insights into the spatial arrangement of atoms, secondary structure elements, and the effects of mutations on protein conformation. The visualization will leverage tools such as `nglview` for interactive 3D rendering.

In [ ]:
# Install dependencies if needed
!pip install py3Dmol==2.4.2 ipywidgets==8.1.7

In [ ]:
# Load Croissant dataset and map to local file paths
from pathlib import Path
import json

import os

# Get the distribution metadata directly
variant_map = {}

for dist in dataset.metadata.distribution:
    if dist.name and dist.name.endswith('.pdb') and dist.content_url:
        parts = dist.id.split("/")[-2:]  # e.g., ["1step_AF", "T146A"]
        label = f"{parts[0]}/{parts[1]}"
        variant_map[label] = dist.content_url

print(f"Mapped {len(variant_map)} variants to remote URLs.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import py3Dmol
import requests
import pandas as pd

# --- Step 1: Build mutation_map with full variant labels like "1step_AF/A103C" ---
subset_to_df_key = {
    "1step_AF": "https://sen.science/doi/10.71728/hw56-vj34/record-sets/af_wuhan_1step_v1",
    "BA1_AF":   "https://sen.science/doi/10.71728/hw56-vj34/record-sets/af_ba1_100",
    "BA2_AF":   "https://sen.science/doi/10.71728/hw56-vj34/record-sets/af_ba2_100",
    "1step_ESM":"https://sen.science/doi/10.71728/hw56-vj34/record-sets/esm_wuhan_1step_v1",
    "BA1_ESM":  "https://sen.science/doi/10.71728/hw56-vj34/record-sets/esm_ba1_100",
    "BA2_ESM":  "https://sen.science/doi/10.71728/hw56-vj34/record-sets/esm_ba2_100",
}

mutation_map = {}
for label in variant_map.keys():
    try:
        subset, seq_id = label.split("/")
        df = dataframes.get(subset_to_df_key.get(subset))
        if df is not None:
            df["decoded_seq_id"] = df["seq_id"].apply(lambda x: x.decode() if isinstance(x, bytes) else x)
            match = df[df["decoded_seq_id"] == seq_id]
            if not match.empty and "site" in match.columns:
                mutation_map[label] = int(match["site"].values[0])
    except Exception as e:
        print(f"\u26a0\ufe0f Error processing {label}: {e}")

# --- Step 2: Setup UI ---
sorted_variants = sorted(variant_map.keys())

search_box = widgets.Text(
    placeholder="Type to filter variants...",
    description="Search:",
    layout=widgets.Layout(width='50%')
)
variant_dropdown = widgets.Dropdown(
    options=sorted_variants,
    value=sorted_variants[0],
    description='Variant:',
    layout=widgets.Layout(width='50%')
)
style_dropdown = widgets.Dropdown(
    options=['cartoon', 'sticks', 'surface', 'spheres', 'lines'],
    value='cartoon',
    description='Style:',
    layout=widgets.Layout(width='30%')
)
output = widgets.Output()

# --- Step 3: Filter dropdown based on search box ---
def filter_dropdown(change):
    term = change['new'].lower()
    matches = [v for v in sorted_variants if term in v.lower()]
    variant_dropdown.options = matches if matches else ["No matches"]
    if matches:
        variant_dropdown.value = matches[0]

search_box.observe(filter_dropdown, names='value')

# --- Step 4: Show structure ---
def show_structure(change=None):
    output.clear_output(wait=True)
    label = variant_dropdown.value
    style = style_dropdown.value

    if label not in variant_map:
        with output:
            print(f"Variant '{label}' not found.")
        return

    try:
        response = requests.get(variant_map[label], timeout=15)
        response.raise_for_status()
        pdb_str = response.text
    except Exception as e:
        with output:
            print(f"Error fetching PDB: {e}")
        return

    # Create a fresh viewer on every call to avoid JS command accumulation
    view = py3Dmol.view(width=700, height=450)
    view.addModel(pdb_str, 'pdb')

    style_map = {
        'cartoon': {'cartoon': {'color': 'spectrum'}},
        'sticks':  {'stick': {}},
        'spheres': {'sphere': {}},
        'lines':   {'line': {}},
        'surface': {'surface': {'opacity': 0.85}},
    }
    view.setStyle(style_map.get(style, {'cartoon': {'color': 'spectrum'}}))

    # Highlight the mutated residue in red
    mut_pos = mutation_map.get(label)
    if mut_pos is not None:
        view.setStyle({'resi': str(mut_pos)}, {'stick': {'color': 'red'}})

    view.zoomTo()

    with output:
        display(view)

# --- Step 5: Bind events and display UI ---
variant_dropdown.observe(show_structure, names='value')
style_dropdown.observe(show_structure, names='value')

display(widgets.VBox([
    search_box,
    widgets.HBox([variant_dropdown, style_dropdown]),
    output
]))

show_structure()  # Initial render

# Observations

This notebook provides a comprehensive exploratory analysis of structural and functional features associated with SARS-CoV-2 RBD mutations. Key observations include:

1. **Dataset Overview**:
    - The dataset comprises multiple RecordSets, each capturing detailed metrics on protein mutations, including structural stability, binding affinity, and expression levels.
    - Metadata and RecordSet descriptions highlight the dataset's focus on ACE-2 binding and RBD expression predictions.

2. **Data Exploration**:
    - Summary statistics reveal variability in structural and functional metrics, such as RMSD, TM-score, SASA, and binding energy, across mutations.
    - Distribution analysis identifies patterns and outliers, providing insights into the diversity of mutation impacts.

3. **Correlation Analysis**:
    - Correlation heatmaps uncover relationships between features, such as strong positive correlations among structural metrics and weak correlations for binding-related metrics.

4. **Visualization**:
    - Histograms and heatmaps effectively illustrate feature distributions and interdependencies, aiding in pattern recognition and hypothesis generation.

5. **Insights**:
    - Structural metrics like `plddt` and `tm_score` indicate high confidence in predictions, while functional metrics such as `expr` and `delta_expr` highlight the impact of mutations on protein expression.
    - Observations of outliers and missing data suggest areas for further investigation or preprocessing.

These analyses provide a foundation for understanding the structural and functional consequences of SARS-CoV-2 RBD mutations, supporting downstream modeling and research efforts. The notebook demonstrates the utility of FAIR² datasets in enabling reproducible and actionable insights into protein mutation impacts.

# Conclusions
In this notebook, we conducted a comprehensive exploratory analysis of structural and functional features associated with SARS-CoV-2 RBD mutations. By leveraging FAIR² datasets, we examined key metrics such as RMSD, TM-score, SASA, binding energy, and expression levels across multiple RecordSets. Through summary statistics, distribution visualizations, and correlation analyses, we identified patterns, variability, and relationships among features, providing insights into the structural stability, binding affinity, and functional impacts of mutations. These findings contribute to a deeper understanding of protein mutation effects, supporting future research and predictive modeling efforts.